In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
import os
import torch
import torch.nn as nn
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score
from torch.utils.data import Dataset, DataLoader
from transformers import (
    AutoTokenizer, 
    AutoModelForSequenceClassification, 
    AdamW, 
    get_linear_schedule_with_warmup
)

# 1. State Chosen Model Required by Competition Rules
MODEL_PATH = "google/rembert"  # Or your local Kaggle input dataset path for rembert

print(f"=== Selected Model: {MODEL_PATH} ===")

def seed_everything(seed=42):
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

seed_everything(42)

CONFIG = {
    "model_name": MODEL_PATH,
    "max_len": 256,
    "batch_size": 8,                  # Strict limit of 8 for RemBERT
    "grad_accum_steps": 2,            # Effective batch size = 16
    "epochs": 3,
    "lr": 1.5e-5,                     # Slightly lower LR for RemBERT stability
    "device": torch.device("cuda" if torch.cuda.is_available() else "cpu"),
    "num_classes": 3,
    "n_splits": 5
}

LABEL_MAP = {"BENIGN": 0, "JAILBREAK": 1, "PROMPT_INJECTION": 2}
INV_LABEL_MAP = {v: k for k, v in LABEL_MAP.items()}

# Dataset Wrapper
class TextDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = str(self.texts[idx])
        encoding = self.tokenizer(
            text,
            truncation=True,
            padding="max_length",
            max_length=self.max_len,
            return_tensors="pt"
        )
        item = {
            "input_ids": encoding["input_ids"].squeeze(0),
            "attention_mask": encoding["attention_mask"].squeeze(0)
        }
        if self.labels is not None:
            item["labels"] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item

# Compute Class Weights for Imbalance (62% BENIGN, 31% JAILBREAK, 7% INJECTION)
def get_class_weights(df):
    counts = df["label_id"].value_counts().sort_index().values
    total = len(df)
    weights = total / (len(counts) * counts)
    return torch.tensor(weights, dtype=torch.float)

# Training Loop with Gradient Accumulation
def train_epoch(model, dataloader, optimizer, scheduler, criterion, device, accum_steps):
    model.train()
    total_loss = 0
    optimizer.zero_grad()
    
    for i, batch in enumerate(dataloader):
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        loss = criterion(outputs.logits, labels)
        
        # Normalize loss for accumulation
        loss = loss / accum_steps
        loss.backward()

        if (i + 1) % accum_steps == 0 or (i + 1) == len(dataloader):
            nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad()

        total_loss += loss.item() * accum_steps
    return total_loss / len(dataloader)

# Validation Loop
def eval_model(model, dataloader, device):
    model.eval()
    preds, targets = [], []
    with torch.no_grad():
        for batch in dataloader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            
            logits = outputs.logits
            preds.extend(torch.argmax(logits, dim=1).cpu().numpy())
            if "labels" in batch:
                targets.extend(batch["labels"].cpu().numpy())
                
    return np.array(preds), np.array(targets)

# --- EXECUTION ---
def run_pipeline():
    train_df = pd.read_csv("train.csv")
    test_df = pd.read_csv("test.csv")
    
    train_df["label_id"] = train_df["label"].map(LABEL_MAP)
    # Stratify across both class label and language
    train_df["strat_key"] = train_df["label"] + "_" + train_df["language"]

    tokenizer = AutoTokenizer.from_pretrained(CONFIG["model_name"])
    
    class_weights = get_class_weights(train_df).to(CONFIG["device"])
    criterion = nn.CrossEntropyLoss(weight=class_weights)

    skf = StratifiedKFold(n_splits=CONFIG["n_splits"], shuffle=True, random_state=42)
    test_dataset = TextDataset(test_df["text"].values, None, tokenizer, CONFIG["max_len"])
    test_loader = DataLoader(test_dataset, batch_size=CONFIG["batch_size"], shuffle=False)
    
    test_preds_folds = np.zeros((len(test_df), CONFIG["num_classes"]))

    for fold, (train_idx, val_idx) in enumerate(skf.split(train_df, train_df["strat_key"])):
        print(f"\n--- Fold {fold + 1} / {CONFIG['n_splits']} ---")
        train_data = train_df.iloc[train_idx]
        val_data = train_df.iloc[val_idx]

        train_ds = TextDataset(train_data["text"].values, train_data["label_id"].values, tokenizer, CONFIG["max_len"])
        val_ds = TextDataset(val_data["text"].values, val_data["label_id"].values, tokenizer, CONFIG["max_len"])

        train_loader = DataLoader(train_ds, batch_size=CONFIG["batch_size"], shuffle=True)
        val_loader = DataLoader(val_ds, batch_size=CONFIG["batch_size"], shuffle=False)

        model = AutoModelForSequenceClassification.from_pretrained(
            CONFIG["model_name"], num_labels=CONFIG["num_classes"]
        ).to(CONFIG["device"])

        total_steps = (len(train_loader) // CONFIG["grad_accum_steps"]) * CONFIG["epochs"]
        optimizer = AdamW(model.parameters(), lr=CONFIG["lr"], weight_decay=0.01)
        scheduler = get_linear_schedule_with_warmup(
            optimizer, num_warmup_steps=int(total_steps * 0.1), num_training_steps=total_steps
        )

        for epoch in range(CONFIG["epochs"]):
            loss = train_epoch(model, train_loader, optimizer, scheduler, criterion, CONFIG["device"], CONFIG["grad_accum_steps"])
            val_preds, val_targets = eval_model(model, val_loader, CONFIG["device"])
            macro_f1 = f1_score(val_targets, val_preds, average="macro")
            print(f"Epoch {epoch + 1}/{CONFIG['epochs']} | Train Loss: {loss:.4f} | Val Macro F1: {macro_f1:.4f}")

        # Fold Predictions on Test Set
        model.eval()
        fold_test_logits = []
        with torch.no_grad():
            for batch in test_loader:
                input_ids = batch["input_ids"].to(CONFIG["device"])
                attention_mask = batch["attention_mask"].to(CONFIG["device"])
                logits = model(input_ids=input_ids, attention_mask=attention_mask).logits
                fold_test_logits.append(torch.softmax(logits, dim=1).cpu().numpy())
        
        test_preds_folds += np.vstack(fold_test_logits) / CONFIG["n_splits"]

    # Generate Final Output File
    final_preds = np.argmax(test_preds_folds, axis=1)
    submission = pd.DataFrame({
        "id": test_df["id"],
        "label": [INV_LABEL_MAP[p] for p in final_preds]
    })
    submission.to_csv("submission.csv", index=False)
    print("\nsubmission.csv created successfully!")

if __name__ == "__main__":
    run_pipeline()